# Escaping the Morlocks · Lab 03: Apache Hudi — follow the completed timeline

> An event timestamp is not a Hudi instant.

## Mission

Replay the same baseline/upsert/delete story as Delta and Iceberg. Capture Hudi instants and read each historical state with `as.of.instant`.

## Same incident, same expected states

| State | Mutation | Rows |
|---|---|---:|
| Baseline | Ada + Babbage | 2 |
| Late arrival | Update Babbage; append Larry with an older `event_time` | 3 |
| Correction | Delete Ada | 2 |

Each run creates a random temporary table. No Docker, named production table, remote catalog or Databricks workspace is required for these local OSS labs. Install `requirements.txt`; Spark downloads its pinned connector packages from Maven Central on first run.

## 1 · Start Hudi 0.15.0 on Spark 3.5.9

Use a Copy-on-Write table so the three labs compare simple committed snapshots. MOR needs a separate read-optimized vs snapshot experiment.

In [ ]:
from pyspark.sql import SparkSession
HUDI="0.15.0"
b=(SparkSession.builder.master("local[2]").appName("Morlocks-Hudi").config("spark.ui.enabled","false").config("spark.sql.shuffle.partitions","2").config("spark.serializer","org.apache.spark.serializer.KryoSerializer").config("spark.kryo.registrator","org.apache.spark.HoodieSparkKryoRegistrar").config("spark.sql.extensions","org.apache.spark.sql.hudi.HoodieSparkSessionExtension").config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.hudi.catalog.HoodieCatalog").config("spark.jars.packages",f"org.apache.hudi:hudi-spark3.5-bundle_2.12:{HUDI}") )
spark=b.getOrCreate(); spark.sparkContext.setLogLevel("ERROR")
OPTS={"hoodie.table.name":"morlock_events","hoodie.datasource.write.table.type":"COPY_ON_WRITE","hoodie.datasource.write.operation":"upsert","hoodie.datasource.write.recordkey.field":"id","hoodie.datasource.write.precombine.field":"balance","hoodie.datasource.write.hive_style_partitioning":"false"}
print("Spark",spark.version,"Hudi",HUDI)

## 2 · Commit the same three states and capture Hudi instants

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, shutil, tempfile
from pyspark.sql import functions as F, types as T
ROOT=Path(tempfile.mkdtemp(prefix="morlocks-LAB-" )).resolve()
PATH=(ROOT/"events").as_uri()
SCHEMA=T.StructType([T.StructField("id",T.LongType(),False),T.StructField("name",T.StringType(),False),T.StructField("segment",T.StringType(),False),T.StructField("balance",T.DoubleType(),False),T.StructField("event_time",T.TimestampType(),False)])
BASE=[(1,"Ada","bronze",100.,datetime(2026,10,24,8,0)),(2,"Babbage","silver",200.,datetime(2026,10,24,8,5))]
LATE=[(2,"Babbage Lovelace","gold",250.,datetime(2026,10,24,8,5)),(99,"Larry Latecomer","time-traveller",42.42,datetime(2026,10,20,9,15))]
def df(rows): return spark.createDataFrame(rows,SCHEMA)
def proof(frame):
 rows=frame.select("id","name","segment","balance",F.date_format("event_time","yyyy-MM-dd HH:mm:ss").alias("event_time")).orderBy("id").collect()
 raw="\n".join("|".join(str(v) for v in r) for r in rows)
 return len(rows),hashlib.sha256(raw.encode()).hexdigest(),rows
df(BASE).write.format("hudi").options(**OPTS).mode("overwrite").save(PATH)
import re
TIMELINE=ROOT/"events"/".hoodie"
def completed_instants():
 found=set()
 for item in TIMELINE.iterdir():
  name=item.name
  match=re.match(r"^(\d{14,17})",name)
  completed=("__COMPLETED__" in name and any(a in name for a in ("__commit__","__deltacommit__","__replacecommit__"))) or item.suffix in {".commit",".deltacommit",".replacecommit"}
  if item.is_file() and match and completed and "inflight" not in name.lower(): found.add(match.group(1))
 return sorted(found)
def instant(): return completed_instants()[-1]
i0=instant(); df(LATE).write.format("hudi").options(**OPTS).mode("append").save(PATH); i1=instant()
del_schema=T.StructType([T.StructField("id",T.LongType(),False),T.StructField("balance",T.DoubleType(),False)])
spark.createDataFrame([(1,999.)],del_schema).write.format("hudi").options(**{**OPTS,"hoodie.datasource.write.operation":"delete"}).mode("append").save(PATH); i2=instant()
print("Hudi instants:",i0,"->",i1,"->",i2); assert i0<i1<i2

## 3 · Read each instant; compare to the same expected business states

In [ ]:
def at(i): return spark.read.format("hudi").option("as.of.instant",i).load(PATH)
states=[at(i0),at(i1),at(i2)]
for label,state in zip(("baseline","late arrival","correction"),states): print(label,proof(state)[:2])
assert [x.count() for x in states]==[2,3,2]
assert states[0].where("id=99").count()==0 and states[1].where("id=99").count()==1 and states[2].where("id=1").count()==0
assert states[1].where("id=99").select("event_time").first()[0].date().isoformat()=="2026-10-20"
print("PASS: Larry's older event_time first appears at later Hudi instant",i1)

## Conclusions · Hudi

**Strength:** completed timeline actions define consistent table states; Hudi supports point-in-time reads. COW offers straightforward snapshot reads; MOR can trade read/write behavior with different query modes.

**Pitfalls:** instant IDs are Hudi-specific. Cleaner policy controls retained file slices. MOR snapshot and read-optimized results may differ around delta commits; this COW notebook does not certify MOR.

**Best practice:** record table identity + completed instant + query mode; test the exact table type and cleaner policy.

In [ ]:
spark.stop(); shutil.rmtree(ROOT,ignore_errors=True); print("PASS: temporary lab cleaned up")